# NSR 전사 서버 — 구글 콜랩판

폰 대신 콜랩의 무료 GPU가 전사합니다. 폰보다 수십 배 빠릅니다.

**쓰는 법 (처음 3분)**

1. 위 메뉴 **런타임 → 런타임 유형 변경**에서 **T4 GPU** 를 고르십시오.
2. **런타임 → 모두 실행**을 누르십시오. 첫 실행은 모델을 받느라 2~5분 걸립니다.
3. 마지막 셀 출력에 나오는 **주소를 복사**해서, NSR 앱의
   **설정 → 전사 모델 → 노트북·서버로 전사**를 켜고 **주소 칸**에 붙여넣으십시오.
   **모델 칸은 비워 두십시오** — 이 서버는 아래에서 고른 모델로 고정됩니다.

그 다음부터는 앱에서 평소처럼 전사를 누르면 콜랩이 대신 일합니다.
30분 조각 기준 대략 1~3분입니다(추정 — 세션마다 다릅니다).

**알고 쓰십시오**

- 기록 음성 **원본이 구글(콜랩) 서버와 Cloudflare 터널을 지나갑니다.**
  내 컴퓨터가 아닙니다. 이 경로가 싫으면 폰 전사나 내 노트북 서버를 쓰십시오.
- 주소 끝에 무작위 비밀 문자열이 붙어 있어서, 주소를 통째로 모르는 남은 못 씁니다.
  그래도 주소를 다른 곳에 붙여넣지 마십시오.
- 콜랩 무료 세션은 탭을 닫거나 오래 놔두면 꺼집니다. **전사하는 동안 이 탭을
  열어 두십시오.** 꺼졌으면 '모두 실행'을 다시 — 주소가 새로 나오니 앱에도 다시 넣습니다.
- 전사할 때만 켜는 개인용입니다. 상시 서버로 두는 것은 콜랩 이용 규칙과 맞지 않습니다.
- 이 노트가 고쳐지면 **위 깃허브 링크로 새로 열어야** 최신판입니다.
  드라이브에 저장해 둔 사본은 옛 판 그대로입니다.


In [ ]:
# 필요한 것 설치 + 터널 프로그램 받기 (1~2분)
# nvidia-cudnn/cublas 를 같이 까는 이유: 콜랩 기본 환경의 cuDNN 판이
# faster-whisper(ctranslate2)와 어긋나면 첫 전사에서 파이썬이 통째로
# 죽는다("kernel restarted") — 실사용에서 그대로 재현된 사고다.
%pip -q install faster-whisper fastapi uvicorn python-multipart nvidia-cudnn-cu12 nvidia-cublas-cu12
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("설치 끝. 다음 셀로.")


In [ ]:
# 전사 서버 — 접수하고(202) 뒤에서 돌리고, 앱이 몇 초마다 결과를 물어간다.
#
# 왜 비동기인가: 다 될 때까지 한 요청으로 기다리는 방식은 앱의 업로드
# 클라이언트(60초)와 Cloudflare 터널(약 100초)이 먼저 끊는다 — 실기기
# 타임아웃으로 재현된 사실이다. 긴 기록일수록 전사가 몇 분씩 걸리므로
# "접수증(job_id) → 진행 조회 → 완성본 수령" 구조여야 한다.
import os
import tempfile
import threading
import traceback
import uuid

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse


def build_app(model, secret: str, log=print) -> FastAPI:
    # log: 백그라운드 스레드의 print 는 셀이 끝난 뒤에는 콜랩 화면에 안
    # 보인다. 실행 셀이 큐를 비우며 대신 찍도록 콜백으로 받는다.
    app = FastAPI()
    jobs = {}
    gpu_lock = threading.Lock()  # GPU 는 하나 — 작업을 줄 세운다.

    def run_job(job_id, path, language, temperature, prompt):
        try:
            with gpu_lock:
                jobs[job_id]["status"] = "processing"
                segment_iter, info = model.transcribe(
                    path,
                    language=language or "ko",
                    temperature=temperature,
                    initial_prompt=prompt or None,
                    beam_size=5,
                    # 잡음·무음 구간에서 같은 문장이 반복되는 환각을 줄인다.
                    condition_on_previous_text=False,
                )
                segments = []
                for i, s in enumerate(segment_iter):
                    segments.append(
                        {"id": i, "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text}
                    )
                    if info.duration:
                        jobs[job_id]["progress"] = min(s.end / info.duration, 1.0)
                jobs[job_id]["result"] = {
                    "task": "transcribe",
                    "language": info.language,
                    "duration": round(info.duration, 2),
                    "text": "".join(s["text"] for s in segments).strip(),
                    "segments": segments,
                }
                jobs[job_id]["status"] = "done"
                log(f"전사 끝({job_id[:8]}): {len(segments)}문장 / {round(info.duration)}초")
        except Exception:
            trace = traceback.format_exc()
            log(trace)
            jobs[job_id]["error"] = trace[-1500:]
            jobs[job_id]["status"] = "error"
        finally:
            os.unlink(path)

    @app.get(f"/{secret}/health")
    def health():
        return {"status": "ok"}

    @app.post(f"/{secret}/v1/audio/transcriptions")
    async def transcribe(
        file: UploadFile = File(...),
        language: str = Form("ko"),
        temperature: float = Form(0.0),
        prompt: str = Form(""),
        response_format: str = Form("verbose_json"),
        model_name: str = Form("", alias="model"),  # 앱이 보내지만 이 서버는 모델 고정
    ):
        suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(await file.read())
            path = f.name
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "progress": 0.0}
        threading.Thread(
            target=run_job, args=(job_id, path, language, temperature, prompt), daemon=True
        ).start()
        log(f"전사 접수({job_id[:8]}): {file.filename}")
        return JSONResponse(status_code=202, content={"job_id": job_id, "status": "queued"})

    @app.get(f"/{secret}/v1/audio/transcriptions/{{job_id}}")
    def job_status(job_id: str):
        job = jobs.get(job_id)
        if job is None:
            return JSONResponse(
                status_code=404,
                content={"error": "모르는 작업입니다. 콜랩 세션이 재시작됐으면 전사를 다시 시작하십시오."},
            )
        if job["status"] == "done":
            return {"status": "done", "result": job["result"]}
        if job["status"] == "error":
            return {"status": "error", "error": job["error"]}
        return {"status": job["status"], "progress": round(job.get("progress", 0.0), 3)}

    return app


In [ ]:
# 모델을 싣고 서버·터널을 띄운다. 마지막에 나오는 주소를 앱에 넣으면 된다.
import ctypes
import glob
import re
import secrets
import subprocess
import threading
import time

import numpy as np
import psutil

# cuDNN/cuBLAS 를 먼저 손으로 적재한다. 콜랩 기본 환경의 판과 어긋나면
# 첫 전사에서 파이썬이 통째로 죽는데("kernel restarted", 추적도 안 남는다),
# 방금 설치한 판을 절대 경로로 미리 올려 두면 이후 탐색이 이쪽을 쓴다.
for pattern in (
    "/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib/libcublas*.so*",
    "/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib/libcudnn*.so*",
):
    for lib in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib)
        except OSError:
            pass

import ctranslate2
import uvicorn
from faster_whisper import WhisperModel

# 공식 CT2 float16 판(1.6GB) — 변환 없이 그대로 실려 콜랩 무료 램(12.7GB)
# 에서 안전하다. 한국어 파인튜닝판(ghost613/faster-whisper-large-v3-turbo-korean)
# 은 float32 3.2GB 라 적재 변환 때 램을 다 먹고 세션이 통째로 죽었다
# ("사용 가능한 모든 RAM을 사용" — 실사고). 큰 램 런타임에서만 바꿔 쓰십시오.
MODEL_ID = "Systran/faster-whisper-large-v3-turbo"
PORT = 8000

gpu = ctranslate2.get_cuda_device_count() > 0
if not gpu:
    print("⚠ GPU 가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤")
    print("  '모두 실행'을 다시 하십시오. CPU 로도 되지만 몇 배 느립니다.")
print(f"모델 여는 중: {MODEL_ID} (세션 첫 실행에서만 내려받습니다)")
model = WhisperModel(
    MODEL_ID,
    device="cuda" if gpu else "cpu",
    compute_type="float16" if gpu else "int8",
)

# 자가 시험: 주소를 내주기 전에 1초짜리 무음을 전사해 본다.
# GPU 경로가 죽을 거라면 앱이 기다리다 타임아웃 나는 대신 여기서 바로 죽어
# 원인이 이 셀에 보인다. 통과하면 실전도 같은 경로다.
print("자가 전사 시험 중...")
list(model.transcribe(np.zeros(16000, dtype=np.float32), language="ko")[0])
print("자가 전사 시험 통과 — 전사 경로 정상.")
vm = psutil.virtual_memory()
print(f"메모리 {vm.used / 1e9:.1f} / {vm.total / 1e9:.1f} GB 사용 중 — 10GB 를 넘어가면 위험하다.")

secret = secrets.token_urlsafe(12)
# 백그라운드 스레드의 print 는 셀이 끝나면 화면에 안 보인다. 큐에 쌓고
# 아래 상주 루프가 대신 찍는다 — 접수/완료 로그가 항상 눈에 보이게.
events = []
app = build_app(model, secret, log=events.append)
threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("터널 주소를 못 받았습니다. 이 셀만 한 번 더 실행해 보십시오.")

print()
print("=" * 62)
print("NSR 앱에 넣을 주소 — 설정 → 전사 모델 → 노트북·서버로 전사:")
print()
print(f"    {url}/{secret}")
print()
print("모델 칸은 비워 두십시오. 이 탭을 닫으면 서버도 꺼집니다.")
print("=" * 62)
print()
print("이 셀은 계속 실행 중인 것이 정상입니다 — 전사 접수/완료 로그가 아래에 찍힙니다.")
while True:
    time.sleep(2)
    while events:
        print(events.pop(0), flush=True)
